# Custom CNN architecture

Build and check the four-class CNN. This notebook does not train the model. It imports make_dataset from the shared src/data_pipeline.py script; keep the src folder in the repository. Uses the fixed ResNet50 shared split. Train a fresh model on this split.

In VS Code, select the repository `.venv` Python kernel and run cells from top to bottom. The shared src/data_pipeline.py helper is retained for model.ipynb.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "dataset").is_dir() and (p / "src").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the project repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository:", ROOT)


## Architecture
The implementation lives in `src/model.py` and is imported below.

RGB 224×224 input → training augmentation → convolution blocks with 32, 64, 128, and 256 filters → global average pooling → Dense(256, ReLU) → Dropout(0.5) → Dense(4, softmax).

Blocks use batch normalization, 2×2 max pooling, and dropout. Adam starts at 0.001; sparse categorical cross-entropy matches integer manifest labels. Inputs are already normalized by the data pipeline.


In [ ]:
import tensorflow as tf



In [ ]:
from src.model import build_model, SEED


## Run the check


In [ ]:
from src.data_pipeline import make_dataset

tf.keras.utils.set_random_seed(SEED)
model = build_model()
model.summary()

# Verify inference using two training images.
images, labels = next(iter(make_dataset("train")))
probabilities = model(images[:2], training=False)

tf.debugging.assert_shapes([(probabilities, (2, 4))])
tf.debugging.assert_all_finite(
    probabilities, "Model produced invalid probabilities"
)
tf.debugging.assert_near(
    tf.reduce_sum(probabilities, axis=1),
    tf.ones(2),
    atol=1e-5,
)

print("\nOutput shape:", probabilities.shape)
print("Probability sums:", tf.reduce_sum(probabilities, axis=1).numpy())
print("Model check completed. Training has not started.")